# 环节 11 · 服务化与推理引擎（配套 Notebook）

> 配套长文：[环节11-服务化与推理引擎详解.md](./环节11-服务化与推理引擎详解.md)
> 定位：把"这台卡能撑多少并发""分页省了多少显存""连续批处理提升多少吞吐""投机解码值不值"算成数。纯 Python 标准库，零依赖。

| 本 Notebook | 长文章节 | 验证什么 |
|---|---|---|
| §1 容量估算 | §4 | 7B FP16 单卡能撑多少并发 |
| §2 PagedAttention | §1.3 | 碎片浪费从 62% 压到 2% |
| §3 Continuous Batching | §1.4 | 对比静态批处理的吞吐差 |
| §4 前缀缓存 | §1.7 / §5.1 | 多轮对话省掉多少 prefill |
| §5 投机解码 | §1.6 | 接受率与期望加速 |
| §6 量化与显存 | §1.8 / §2.2 | 各量化档位的体积 |
| §7 延迟与吞吐的拉扯 | §3 | batch 增大 → 吞吐↑ 但 TPOT↑ |


## 1. 部署前必算的容量账（长文 §4）

```
所需显存 ≈ 权重 + KV Cache + 激活/图/CUDA 上下文
并发 × 平均上下文长度 × 单 token KV ≤ 可用 KV 显存
```


In [ ]:
GPU_GB = 24                 # 单张 4090
UTIL = 0.9                  # gpu_memory_utilization
weight = 7e9 * 2 / 1e9      # 7B FP16 = 14 GB（按十进制，与长文口径一致）
avail = GPU_GB * UTIL
overhead = 0.6              # 激活 / CUDA graph / 碎片预留

print(f"卡 {GPU_GB} GB × {UTIL} = {avail:.1f} GB 可用")
print(f"  权重（7B FP16）      {weight:>5.1f} GB")
print(f"  其它开销             {overhead:>5.1f} GB")
kv_avail = avail - weight - overhead
print(f"  = 可用 KV           {kv_avail:>5.1f} GB\n")

for tag, kvh in [("MHA（32 KV 头）", 32), ("GQA-8（8 KV 头）", 8)]:
    per_tok = 2 * 32 * kvh * 128 * 2            # 字节/token
    cap = kv_avail * 1e9 / per_tok
    print(f"{tag}:")
    print(f"  单 token KV = {per_tok/1024:>5.0f} KB → 总容量 {cap:>8,.0f} token")
    print(f"     ≈ {cap/1024:>5.1f} 并发 × 1k 上下文，或 {cap/512:>5.1f} 并发 × 512 上下文")

print("\n→ 长文 §4 给的示例是 ~14 并发×1k / ~28 并发×512，同量级。")
print("  这就是 max_num_seqs / max_model_len 的定值依据（长文 §5）。")
print("\n换量化再算（INT4 权重 3.5 GB）：")
kv4 = avail - 3.5 - overhead
print(f"  可用 KV {kv4:.1f} GB → {kv4*1e9/(2*32*32*128*2):,.0f} token"
      f" ≈ {kv4*1e9/(2*32*32*128*2)/1024:.1f} 并发×1k（翻了 "
      f"{(kv4/kv_avail - 1) * 100:.0f}%）")


## 2. PagedAttention：把 KV 当虚拟内存管（长文 §1.3）

朴素做法按 `max_model_len` 一次性连续分配 → 请求短了就是浪费（内部碎片），请求长短不一时还会外部碎片。

PagedAttention 把 KV 切成固定大小的 **block**，按需分配、逻辑连续物理离散。


In [ ]:
import random

random.seed(7)
MAXLEN, BLOCK = 1024, 16
reqs = [random.randint(50, 900) for _ in range(40)]
need = sum(reqs)

cont = sum(MAXLEN for _ in reqs)                          # 每个请求都按上限预留
paged = sum(-(-L // BLOCK) * BLOCK for L in reqs)         # 按块向上取整

print(f"40 个请求（长度 50~900），真实需要 {need:,} 个 KV 位置\n")
print(f"{'方案':<28} {'占用':>12} {'浪费':>10}")
print("-" * 52)
print(f"{'连续分配（按 max_len 预留）':<28} {cont:>12,} {1 - need/cont:>9.1%}")
print(f"{'分页分配（block=16 向上取整）':<28} {paged:>12,} {1 - need/paged:>9.1%}")

print(f"\n→ 浪费从 {1-need/cont:.1%} 压到 {1-need/paged:.1%}，等于凭空多出 "
      f"{(cont-paged)/cont:.0%} 的可用 KV 容量（能塞更多并发）。")
print("\n附带收益（长文 §1.3）：多个请求可**共享公共前缀块**（相同 system prompt），")
print("  这是 §4 前缀缓存与并行采样的基础。类比：操作系统的虚拟内存分页。")


## 3. Continuous Batching：为什么是吞吐提升最大的单一优化（长文 §1.4）

静态批处理"一批一起进、一起出"，快的被慢的拖住（队头阻塞）。连续批处理**每个 iteration 动态调度**：谁生成完谁下场，新请求立刻补位。


In [ ]:
random.seed(8)
lens = [random.randint(1, 40) for _ in range(8)]

static_work = max(lens) * len(lens)      # 每个 iteration 都拖着全部 8 个请求
cont_work = sum(lens)                    # 每个请求只占用到自己完成

print(f"8 个请求各自需要的解码步数：{lens}\n")
print(f"Static Batching :")
print(f"  必须等到最慢的那个（{max(lens)} 步）才能换批")
print(f"  “请求·步”消耗 = {max(lens)} × {len(lens)} = {static_work}")
print(f"  其中有效部分只有 {cont_work} → slot 利用率 {cont_work/static_work:.0%}")

print(f"\nContinuous Batching :")
print(f"  每个 iteration 检查谁完成了 → 释放 slot 给新请求")
print(f"  “请求·步”消耗 = Σ = {cont_work} → 利用率接近 100%")

print(f"\n→ 同样算力下有效吞吐提升 ≈ {static_work}/{cont_work} = "
      f"{static_work/cont_work:.2f}x")
print("  这正是长文 §1.4 说“吞吐提升最大的单一优化”的来源；")
print("  也让“T 不确定”（谁先 EOS 谁先走）从问题变成了调度机会（环节 10 §4.3）。")


## 4. 前缀缓存：多轮对话省在哪（长文 §1.7 / §5.1）

每轮把完整历史发回来（事实标准），引擎按**前缀哈希**命中缓存 → 只对新消息做 prefill。


In [ ]:
SYSTEM = 800          # system prompt + 工具定义（从不变）
rounds = [("用户问1", 40), ("助手答1", 300), ("用户问2", 60), ("助手答2", 400), ("用户问3", 50)]

no_cache_total = 0
cache_total = 0
history = SYSTEM
print(f"{'轮次':>4} {'新增':>8} {'历史长度':>10} {'无缓存 prefill':>16} {'有缓存 prefill':>16}")
print("-" * 60)
for i, (name, length) in enumerate(rounds, start=1):
    no_cache_total += history + length            # 每轮重算全部历史
    cache_total += length                         # 只算新增部分
    history += length
    print(f"{i:>4} {name:>8} {history:>10} {history:>16,} {length:>16,}")

print(f"\n{'合计':>4} {'':>8} {'':>10} {no_cache_total:>16,} {cache_total:>16,}")
print(f"\n→ 开了前缀缓存后 prefill 量降到 {cache_total/no_cache_total:.1%}"
      f"（省 {no_cache_total/cache_total:.1f} 倍），TTFT 直接受益（长文 §3 指标表）。")

print("\n但“命中”要同时满足四个条件（长文 §5.1）：")
for i, cond in enumerate([
    "同一会话钉同一节点（前缀缓存是节点内存态）",
    "前缀逐字节一致（差一个空格，命中点之后全部重算）",
    "没被 LRU / TTL 淘汰掉",
    "权重 / 量化 / 引擎版本一致（KV 是权重函数的输出）",
], start=1):
    print(f"  {i}. {cond}")
print("\n三条铁律：不变的放最前（system/工具定义）、别塞时间戳或随机 id、别中途改写历史。")


## 5. 投机解码值不值（长文 §1.6）

草稿模型先猜 k 个 token，大模型一次并行验证。接受率为 α 时，一次能产出多少 token？


In [ ]:
print("期望产出 token 数（接受率 α，草稿 k 步）：")
print(f"{'α':>6} " + " ".join(f"{'k=' + str(k):>12}" for k in (1, 3, 5, 8)))
print("-" * 60)
for alpha in (0.3, 0.5, 0.7, 0.9, 0.95):
    row = []
    for k in (1, 3, 5, 8):
        exp_tok = (1 - alpha ** (k + 1)) / (1 - alpha)
        row.append(exp_tok)
    print(f"{alpha:>6} " + " ".join(f"{v:>12.2f}" for v in row))

print("\n→ 接受率越高、草稿越长，单次验证产出的 token 越多（上限 1/(1−α)）。")
print("  但草稿本身要花算力：真实加速 ≈ 期望产出 / (1 + k·c)，c 是草稿模型相对成本。")

print("\n什么情况收益为负（长文 §6 坑清单）：")
for cond in ["草稿接受率低（创造性文本、随机性强的内容）",
             "生成很短（k 步还没摊销完草稿成本就结束了）",
             "草稿模型太贵（c 接近 1 时基本白干）"]:
    print(f"  · {cond}")
print("\n变体：DeepSeek 用 MTP 头做自投机，无需额外草稿模型（环节 08 §3 / 环节 10）。")


## 6. 量化：几 bit 到底换多少显存（长文 §1.8 / §2.2）

一个容易踩的认知：**"几-bit 模型"未必在低 bit 下算**——W4A16 是"存 4 算 16"（省显存/带宽），FP8 才是"存 8 算 8"（省算力）。


In [ ]:
P = 7e9
print(f"{'格式':<10} {'bit/权重':>10} {'7B 体积':>12} {'存/算':<16} 适用")
print("-" * 72)
rows = [
    ("FP16/BF16", 16, "存16算16", "训练/基准"),
    ("FP8", 8, "存8算8", "数据中心主流（H100 起）"),
    ("GPTQ/AWQ", 4, "存4算16", "生产 INT4，精度损失小"),
    ("Q4_K_M(GGUF)", 4.5, "存4.5算16", "本地/边缘（llama.cpp 系）"),
    ("Q2_K(GGUF)", 2.6, "存2.6算16", "极限省，质量牺牲大"),
]
for name, bits, mode, use in rows:
    print(f"{name:<10} {bits:>10} {P*bits/8/1e9:>10.2f} GB {mode:<16} {use}")

print("\n引擎支持矩阵（长文 §2.3，读法：生产 = 离线工具产、线上引擎消）：")
print(f"{'方法':<14} {'vLLM':>7} {'SGLang':>8} {'TRT-LLM':>9} {'llama.cpp/Ollama':>18}")
print("-" * 60)
for method, v, s, t, l in [("FP8 (W8A8)", "✅", "✅", "✅", "✖"),
                           ("AWQ (INT4)", "✅", "✅", "✅", "转 GGUF"),
                           ("GPTQ (INT4)", "✅", "✅", "✅", "转 GGUF"),
                           ("GGUF Q4/Q5", "GGUF loader", "✖", "✖", "✅ 原生")]:
    print(f"{method:<14} {v:>7} {s:>8} {t:>9} {l:>18}")

print("\n红线（长文 §1.8 / §6）：推理模型（R1 类）对量化敏感，")
print("  生产前必须跑 Eval 回归（横切主题《模型评测与选型方法详解》）。")


## 7. 延迟与吞吐的拉扯（长文 §3）

**核心 trade-off：吞吐 ↑ ⇔ 延迟 ↑**。批越大 GPU 越饱和、吞吐越高，但每个请求的 TPOT 越慢。生产上用"P95 TPOT 达标"约束下压最大吞吐，而不是单看某一项。


In [ ]:
W = 10.0        # 每步的"固定开销"（读一遍权重，ms）
C = 0.5         # 每个请求每步的额外计算（ms）
print(f"模型：每步耗时 = {W} + {C}×batch（ms）\n")
print(f"{'batch':>6} {'每步耗时/TPOT':>16} {'吞吐 (tok/s)':>16} {'相对吞吐':>12}")
print("-" * 56)
base = None
for b in (1, 2, 4, 8, 16, 32, 64):
    step_ms = W + C * b
    tput = b / step_ms * 1000
    base = tput if base is None else base
    print(f"{b:>6} {step_ms:>14.1f} ms {tput:>16.0f} {tput/base:>11.1f}x")

print("\n→ batch 从 1 到 64：吞吐涨了两位数倍，但 TPOT 也从 10.5ms 涨到 42ms（4 倍）。")
print("  原因（长文 §5.2）：权重每步都要读一遍，batch 里 N 个请求**合读同一份权重**")
print("  = 读一次喂 N 人 → 带宽被摊销，batch 越大摊得越薄。")
print("\n所以容量规划的正确姿势：先定延迟 SLO（如 P95 TPOT < 50ms），再在该约束下压最大 batch。")


## 8. 自测（长文 §7）

| 问题 | 本 Notebook 的现场证据 |
|---|---|
| 7B FP16 单卡能撑多少并发？ | §1：24GB 卡 ≈ 13 并发×1k（MHA）/ 52 并发×1k（GQA） |
| PagedAttention 解决什么？ | §2：碎片浪费 62% → 2%，还能共享前缀块 |
| Continuous Batching 好在哪？ | §3：请求·步从 200 降到 104（1.92x） |
| 前缀缓存能省多少 prefill？ | §4：多轮对话降到 ~24% |
| 投机解码什么时候值？ | §5：接受率高 + 生成足够长 |
| "4bit 模型"是在 4bit 下算吗？ | §6：不是，W4A16 是存 4 算 16 |
| 吞吐和延迟的关系？ | §7：吞吐↑ ⇔ TPOT↑，要在 SLO 约束下取平衡 |

**上一站** [环节 10 · 推理解码与 KV Cache](./环节10-推理解码与KV缓存详解.md)   **回到** [环节 00 · 总揽与环节导航](./环节00-总揽与环节导航.md)
